# Model bias (mu + bu + bi, teregularisasi)
        **recsys › rating_prediction** · notebook `bias_model`

        **Kapan dipakai:** Baseline kuat yang sering sulit dikalahkan: pengguna ramah/pelit dan film bagus/jelek.

        **Jebakan umum:** Tidak menangkap selera personal (interaksi pengguna-film).

**Data demo:** MovieLens 100k (943 pengguna, 1682 film, 100.000 rating). Evaluasi: **split temporal** (20% rating terbaru sebagai uji). Metrik: RMSE dan MAE.

## 0. Setup
Import dan lokasi proyek. Tidak perlu diubah.

In [ ]:
import sys, pathlib, warnings
ROOT = next(p for p in [pathlib.Path.cwd().resolve(), *pathlib.Path.cwd().resolve().parents]
            if (p / "src" / "superdb").is_dir())
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.max_columns", 50)
from superdb.common import *
from superdb.paths import out_dir

## 1. Konfigurasi

In [ ]:
DATA = "data_latihan/recsys/ml-100k"; MODEL_KEY = "bias_model"
OUT = out_dir("recsys", "rating_prediction", MODEL_KEY); set_seed(42)

## 2. Data dan split temporal
Mengacak rating memberi skor terlalu optimis karena 'mengintip masa depan'. Split temporal meniru kondisi nyata.

In [ ]:
from types import SimpleNamespace
from superdb.recsys import load_ml100k, temporal_split, leave_last_out, interaction_matrix, rating_metrics, rank_metrics
r, movies, users, n_users, n_items = load_ml100k(DATA)
genre_cols = [c for c in movies.columns if c not in ("item", "title", "release_date", "i", "unknown")]
ctx = SimpleNamespace(n_users=n_users, n_items=n_items, movies=movies, users=users, genres=genre_cols, genre_cols=genre_cols)
print(f"{len(r)} rating | {n_users} pengguna x {n_items} film | kerapatan {len(r) / (n_users * n_items):.1%} | rata-rata rating {r.rating.mean():.2f}")

In [ ]:
train, test = temporal_split(r, 0.2)
print(f"latih {len(train)} | uji {len(test)} | uji berisi pengguna/film baru: {int((~test.u.isin(train.u)).sum())} / {int((~test.i.isin(train.i)).sum())} baris (cold-start)")

## 3. Model
Semua model berbentuk `fit_predict(train, test, ctx)` yang mengembalikan prediksi rating untuk baris `test`.

In [ ]:
import numpy as np, pandas as pd
from superdb.recsys import BiasModel, surprise_fit_predict, torch_mf

def fit_predict(train, test, ctx):
    return BiasModel(reg_u=10, reg_i=25).fit(train, ctx.n_users, ctx.n_items).predict(test)

## 4. Evaluasi (dibanding rata-rata global)

In [ ]:
import time; t0 = time.time(); pred = np.asarray(fit_predict(train, test, ctx), float); sec = time.time() - t0
assert pred.shape == (len(test),) and np.isfinite(pred).all()
base = rating_metrics(test.rating, np.full(len(test), train.rating.mean())); m = rating_metrics(test.rating, pred)
display(pd.DataFrame({"model": m, "rata-rata global": base}).T.round(4)); print(f"waktu: {sec:.1f} dtk | perbaikan RMSE terhadap rata-rata global: {100 * (1 - m['rmse'] / base['rmse']):.1f}%")
metrics = {**m, "rmse_global_mean": base["rmse"], "seconds": sec}

## 5. Di mana model salah?
Galat menurut jumlah rating film dan menurut nilai rating asli.

In [ ]:
d = test.assign(pred=np.clip(pred, 1, 5)); d["err"] = (d.rating - d.pred).abs(); d["n_item_train"] = d.i.map(train.groupby("i").size()).fillna(0)
fig, ax = plt.subplots(1, 2, figsize=(9, 3.2))
d.groupby(pd.cut(d.n_item_train, [-1, 0, 5, 20, 100, 1000]), observed=True).err.mean().plot.bar(ax=ax[0], title="MAE vs jumlah rating film di train"); d.groupby("rating").err.mean().plot.bar(ax=ax[1], title="MAE per nilai rating asli")
plt.tight_layout(); plt.savefig(OUT / "errors.png", dpi=130); plt.show()
pd.DataFrame({"user": test.user, "item": test.item, "rating": test.rating, "pred": pred}).to_csv(OUT / "predictions.csv", index=False)
save_json(metrics, OUT / "metrics.json"); print("OK. hasil di", OUT)

## Langkah berikutnya
- RMSE kecil belum tentu rekomendasi bagus: untuk daftar rekomendasi nilai dengan metrik ranking (lihat `top_n/`).
- Rata-rata blend beberapa model (bias + SVD + KNN) biasanya menurunkan RMSE lagi.